# 🏗️ Note — Spark Architecture: what happens when you *submit* a program

This is a **deep-dive note**, not a chapter: no new PySpark functions here, only the machinery underneath every chapter.
It expands the short architecture table at the top of [chapter7.ipynb](chapter7.ipynb) with the full story —
submit, JVM, executors, cores, tasks, success/failure, cluster managers, and the two deployment modes.

Source material: the *Spark Architecture* video of the PySpark playlist
([youtu.be/CYyUuInwgtA](https://youtu.be/CYyUuInwgtA?list=PL2IsFZBGM_IHCl9zhRVC1EXTomkEp_1zm)).

**How this note is arranged — read it top to bottom, once:**

> **Section 0 is the dictionary: every word is defined there and only there.**
> **Sections 1-9 are the story**, and they use those words without stopping to re-explain them.
> **Section 10** places this course's own setups on the map.

👉 The **full architecture figure** (driver → cluster manager → worker nodes → executors → cores) is in
**section 3**; every other figure zooms into one part of it.

| # | Section | Question it answers |
|---|---------|--------------------|
| **0** | **The vocabulary** | **What is a JVM, a driver, a worker node, an executor, a core, a cluster manager? And what do *lazy*, *query plan*, *task*, *stage*, *job*, *Py4J*, *UDF*… mean?** |
| 1 | Submitting a program | What does *"submit"* even mean? |
| 2 | Where Python fits | You write Python but Spark is Scala — which one runs where? |
| 3 | The 6-step lifecycle | Who talks to whom, in what order? |
| 4 | Executors vs cores | Is it 4 executors or 2? (the confusing bit) |
| 5 | "Copying the program" | What actually travels to the executors? |
| 6 | Success or failure | What counts as failure, and what does Spark do about it? |
| 7 | Resource manager | Is it responsible for the driver or the SparkSession? The 4 types. |
| 8 | Client vs cluster mode | Where does the driver *sit*? |
| 9 | spark-submit cheat sheet | Which flag maps to which part of the story? |
| 10 | Where this course sits | `local[*]`, the Docker cluster, Databricks — and how to watch it all in the Spark UI. |

## 0. The vocabulary — defined once, here

Read this section once, in order. Sections 1-9 then tell the *story* and use these words freely — **they never
stop to re-define them**, so this is the only place you ever have to scroll back to.

### 0a. What a "JVM" is — because every Spark process is one

**JVM = Java Virtual Machine** — a program whose job is to run other programs. Java and Scala code is compiled to
**bytecode** (not Windows/Linux machine code), and the JVM executes that bytecode while managing its own chunk of
memory — the **heap** — and cleaning it up (*garbage collection*). **One JVM = one operating-system process**, with
its own memory limit.

Spark's engine is written in **Scala**, so every box in every figure in this note is a JVM: the driver is one, and
each executor is one. `--executor-memory 4g` sets one executor JVM's heap; fill it and you get `OutOfMemoryError`,
kill the JVM and everything cached inside it is gone. Two executors on one machine = two separate JVM processes,
isolated from each other. Where your **Python** sits in that picture is the one PySpark-specific twist — section 2.

### 0b. The cast — the things that exist while your program runs

Sections 1-9 are these five talking to each other. The middle column is the one that matters: each is either a
**machine**, a **process** (a running program the OS schedules), or just a **slot** — mixing those up is what makes
the architecture confusing.

| Cast member | It is a… | What it does |
|---|---|---|
| **Driver** | **process** — the JVM that runs *your* program (in PySpark: your Python process with a JVM beside it). Exactly one per application; in a notebook, the kernel *is* the driver. | The brain. Holds the `SparkSession`, turns your code into a **query plan**, asks for executors, cuts the work into tasks, hands them out, collects the results. It does **not** process the data itself. |
| **Worker node** | **machine** — one physical or virtual computer in the cluster, with CPUs and RAM to lend. | Offers its free cores + memory to the cluster manager and **hosts** the executor processes launched on it. One node can hold executors of several different applications at once. |
| **Executor** | **process** — one JVM on a worker node, belonging to exactly **one** application. | The muscle. Runs the tasks the driver sends it: reads its own slices of the data, computes, caches, sends results back. Dies when the application ends. |
| **Core** | **slot** — a task slot inside an executor (`--executor-cores 2` = 2 slots), not a dedicated physical CPU. | Runs **one task at a time**. 2 cores = that executor chews through 2 partitions in parallel. |
| **Cluster manager** | **process** — a separate, always-running service (standalone Master / YARN / Kubernetes), independent of your app and shared by all of them. | The landlord. Tracks every node's free capacity, and launches and kills executor JVMs on request (section 7). |

The one sentence that chains them together:

> Your **driver** asks the **cluster manager** for resources; the manager starts **executor** JVMs on **worker
> nodes**; each executor's **cores** run **tasks**, one **partition** per task.

**Words that get mixed up** — you will hear all of these, in the video and at work:

- **node vs worker** — the *node* is the machine; strictly, the *worker* is the manager's small agent process
  running on it. Nearly everyone says "worker" for the machine too, and that is harmless.
- **executor vs node** — an executor is one JVM. A node normally runs several.
- **core vs CPU** — a Spark "core" is a task slot, not a physical CPU; in scheduling talk **slot** is the same word.
- **driver vs master** — the driver is *your application's* brain and dies with it; the master (= cluster manager,
  = resource manager: three names, one role) is the cluster's landlord and outlives every app.

Two special cases to keep in the back of your mind:

- **`local[*]` — this whole course** — collapses the cast into one process: the same JVM is driver *and* executor,
  there is no cluster manager and no network. `*` = "use every core on this machine".
- **Cluster deploy mode** seats the driver *inside* the cluster, on a worker node, instead of on your laptop
  (section 8). Same cast, different seat.

### 0c. Jargon buster — every borrowed word, in plain English

| Word | Plain meaning |
|------|---------------|
| **Transformation** | An instruction that only *describes* work: `select()`, `filter()`, `withColumn()`, `join()`, `repartition()`. Running the line does **no work at all**. |
| **Action** | An instruction that *demands an answer*: `show()`, `count()`, `collect()`, `write()`. Only an action starts real work on the cluster. |
| **Lazy** | The name for that behaviour — Spark postpones everything until an action forces it. That delay is what lets Spark optimise the whole chain at once. |
| **Query plan** | The **recipe** Spark builds out of your transformations: a tree of steps ("scan this file → keep these 2 columns → filter salary > 4000 → group by dept"). Your Python code is *not* what runs on the cluster; the plan is. See the four stages below. |
| **Catalyst** | Spark's optimiser — the component that rewrites your plan into a cheaper one (drop unread columns, apply filters before joins, fold constants). |
| **Partition** | One chunk of the rows (chapter 7). |
| **Task** | The unit of work handed to a core: run the plan over **one partition**. |
| **Stage** | A group of tasks with no shuffle in between. |
| **Job** | Everything triggered by **one action**. |
| **Application** | One submitted program = one driver + the executors it holds. Two notebooks open = two applications, each with its own executors. (Nesting of these four: section 6.) |
| **Shuffle** | Physically moving rows between partitions/executors so the ones that belong together end up together (chapter 7). |
| **Py4J** | A small Python library that lets Python call code living in a **separate Java process**. PySpark's plumbing: `emp.filter(...)` in Python sends a message over a local socket to the driver's JVM, which runs the real Scala method and sends the answer back. So a PySpark DataFrame is a thin Python **handle**; the real object lives in the JVM. It is also why Python errors sometimes show a Java stack trace. |
| **UDF** | **U**ser **D**efined **F**unction — *your own* Python function applied to column values, e.g. `udf(lambda s: s.upper())`. Spark's built-ins (`upper()`, `when()`, `to_date()`) run inside the JVM; a Python UDF cannot, so every row is shipped out of the JVM to a python worker process and back. Slow, and Catalyst cannot see inside it to optimise. Use built-ins first; if you truly need one, prefer a **pandas UDF** (vectorised, sends batches via Arrow instead of row by row). |
| **Closure** | Your function *plus the outside variables it uses*. Spark packs the closure up and ships it to the executors — which is why a variable your lambda touches must be serialisable. |
| **Serialisation** | Turning an object into bytes so it can travel over a network or between processes (and back again = deserialisation). Every task, closure, shuffle row and UDF value pays this cost. |

### The four stages of a query plan

```text
 your code:  emp.filter(...).select(...)      nothing runs yet
      │
      ▼
 1. unresolved logical plan  "some filter on some column"
      │   Analyzer: do these columns exist? what are their types?
      ▼
 2. analysed logical plan    "filter on emp.salary : string"
      │   Catalyst: drop unused columns, push filters down
      ▼
 3. optimised logical plan   "read 2 columns, filter early"
      │   Planner: pick the real algorithms (scan, hash join,
      │            exchange = shuffle)
      ▼
 4. physical plan   ->   stages   ->   tasks   ->   executors
```

`df.explain()` prints stage 4; `df.explain(True)` prints all four. The Spark UI's **SQL** tab draws the same
thing as a diagram. You will meet these words again in chapter 7's shuffle discussion and in every tuning article.

## 1. What does "submitting a program" mean?

A Spark program is just a file of code — say `daily_sales.py`. Running it is not like running a normal Python script,
because the code has to end up spread over many machines. **Submitting** is that hand-over step:

> You hand your program to Spark's launcher, and the launcher starts a **driver** process for it and asks a cluster for machines to work on.

The launcher is a command-line tool that ships with Spark: **`spark-submit`**.

```bash
spark-submit \
  --master yarn \             # which cluster manager to talk to
  --deploy-mode client \      # where the driver runs (section 8)
  --num-executors 4 \         # I want 4 executors ...
  --executor-cores 2 \        # ... with 2 cores each = 8 cores
  --executor-memory 4g \      # heap of each executor JVM
  daily_sales.py 2026-08-13   # your program + its arguments
```

Nothing magic happened: `spark-submit` started **one JVM that runs your program** (the driver), and your program's
`SparkSession.builder.getOrCreate()` line then went shopping for executors with the numbers above.

### The three ways you will meet "submit"

| How you run Spark | What "submit" looks like |
|---|---|
| `spark-submit app.py` (production, scheduled jobs, Airflow) | Explicit. You type the command; the driver starts, runs, exits. |
| **A notebook** (this course, Jupyter, Databricks) | Invisible. The notebook kernel *is* the driver process, and it stays alive for hours. The "submit" happens the moment you run `SparkSession.builder...getOrCreate()`; the "exit" happens on `spark.stop()` or when you shut the kernel down. |
| A UI / API (Databricks job, Livy, `spark-submit` REST) | Someone else runs the equivalent of `spark-submit` for you. |

So in these notebooks you are always in the middle of an already-submitted application — which is exactly why
each chapter creates its own `SparkSession` at the top and (from chapter 9 on) calls `spark.stop()` at the bottom.

## 2. Where your Python fits — PySpark is a wrapper

Spark's engine is Scala running in JVMs (section 0a), but you are writing Python. So which of your two languages
runs where?

```text
 ┌────────────────────────────┐
 │ DRIVER                     │
 │  your Python process       │  ← this notebook
 │        │  Py4J socket      │
 │        ▼                   │
 │  JVM: builds + optimises   │
 │       the query plan       │
 └─────────────┬──────────────┘
               │  down: tasks     up: results
 ┌─────────────▼──────────────┐
 │ EXECUTOR  (one JVM)        │
 │  JVM: reads the data and   │
 │       runs the plan on     │
 │        │  its partitions   │
 │        ▼  only for a       │
 │  python worker  Python UDF │
 └────────────────────────────┘
```

- Your Python code only *describes* the work (`emp.filter(...).groupBy(...)`). It travels through the **Py4J**
  bridge to the driver's JVM, which turns it into the **query plan**.
- The plan itself runs in the executor JVMs — **no Python involved**, whatever language you wrote it in. This is
  why PySpark is not "slow Python": your Python never sees a single row of data.
- The exception is a **UDF**: your own Python function can't run inside the JVM, so each executor starts extra
  **python worker** processes and ships rows out to them and back, serialising both ways. That detour is the whole
  reason the built-in functions (`when`, `regexp_replace`, `to_date`, …) beat a Python UDF — and the next cells
  make it visible in a query plan.

In [18]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder.appName("Spark Architecture Notes").master("local[*]").getOrCreate())

In [19]:
# Peek at the machinery of the session we just built
import os

sc = spark.sparkContext

print("Spark version          :", spark.version)
print("Application name       :", sc.appName)
print("Application id         :", sc.applicationId)          # the id the cluster manager tracks
print("master (cluster mgr)   :", sc.master)                 # local[*] = no real cluster
print("deploy mode            :", spark.conf.get("spark.submit.deployMode", "not set (local run)"))
print("default parallelism    :", sc.defaultParallelism)     # = usable cores => tasks that can run at once
print("CPU cores on this box  :", os.cpu_count())
print("Spark UI               :", sc.uiWebUrl)               # Executors / Environment tabs show all of this

# the JVM our Python process is talking to over Py4J
try:
    jvm = sc._jvm.java.lang.System
    print("JVM (java) version     :", jvm.getProperty("java.version"), "-", jvm.getProperty("java.vm.name"))
except Exception as e:
    print("JVM lookup failed      :", e)

Spark version          : 3.3.0
Application name       : Spark Architecture Notes
Application id         : local-1786651654216
master (cluster mgr)   : local[*]
deploy mode            : client
default parallelism    : 20
CPU cores on this box  : 20
Spark UI               : http://0a8b05ba503f:4040
JVM (java) version     : 11.0.25 - OpenJDK 64-Bit Server VM


### Seeing the query plan and a UDF with your own eyes

The two cells below make section 0c's words concrete:

1. Transformations build a **query plan** and run nothing — `explain()` prints the physical plan Spark *would* run.
2. A **built-in** function stays inside the JVM, while a **Python UDF** adds a `BatchEvalPython` step to the plan:
   that step is exactly the "rows leave the JVM, go to a python worker, come back" detour described above.
   Seeing `BatchEvalPython` in a plan is the standard way to spot a slow UDF.

Both cells only *print plans*, so they are safe to run anywhere. Note that actually **executing** a Python UDF
needs a Python version your PySpark build supports — in this repo's local `.venv` (PySpark 3.3.0 on Python 3.12)
the python worker crashes with `Python worker exited unexpectedly`; the Docker Jupyter image in
[docker-images/](docker-images/) runs UDFs fine.

In [20]:
# 1) Transformations are lazy: this cell builds a PLAN, it does not read or compute anything
from pyspark.sql.functions import col, when, lit

df = (spark.range(1, 4)                                    # id = 1, 2, 3
      .withColumn("dept", when(col("id") % 2 == 0, lit("hr")).otherwise(lit("sales")))
      .withColumn("salary", col("id") * 2000))

plan_df = df.filter("salary > 4000").select("dept")   # still nothing has run
print(">>> physical plan of  filter(...).select('dept'):")
plan_df.explain()                                     # the recipe (stage 4 of section 0c)

print(">>> only NOW does a job actually run (an action):")
plan_df.show()

>>> physical plan of  filter(...).select('dept'):
== Physical Plan ==
*(1) Project [CASE WHEN ((id#117L % 2) = 0) THEN hr ELSE sales END AS dept#119]
+- *(1) Filter ((id#117L * 2000) > 4000)
   +- *(1) Range (1, 4, step=1, splits=20)


>>> only NOW does a job actually run (an action):
+-----+
| dept|
+-----+
|sales|
+-----+



In [21]:
# 2) built-in function vs Python UDF, seen in the plan
from pyspark.sql.functions import udf, upper
from pyspark.sql.types import StringType

shout = udf(lambda s: s.upper() + "!", StringType())   # a User Defined Function

print(">>> built-in upper(): runs inside the executor JVM")
df.select(upper("dept")).explain()

print(">>> Python UDF: look for 'BatchEvalPython' = the trip out to a python worker")
df.select(shout("dept")).explain()

>>> built-in upper(): runs inside the executor JVM
== Physical Plan ==
*(1) Project [CASE WHEN ((id#117L % 2) = 0) THEN HR ELSE SALES END AS upper(dept)#132]
+- *(1) Range (1, 4, step=1, splits=20)


>>> Python UDF: look for 'BatchEvalPython' = the trip out to a python worker
== Physical Plan ==
*(2) Project [pythonUDF0#137 AS <lambda>(dept)#135]
+- BatchEvalPython [<lambda>(CASE WHEN ((id#117L % 2) = 0) THEN hr ELSE sales END)#134], [pythonUDF0#137]
   +- *(1) Range (1, 4, step=1, splits=20)




## 3. The 6-step lifecycle of a submitted program

This is the whole story, in the order the video draws it — **the full picture of a submitted application**.
Assume we asked for **4 executors × 2 cores**. (Driver, cluster manager, node, executor, core: section 0b.)

```text
       ┌──────────┐
       │  Driver  │  ← your program + SparkSession (1 per app)
       └────┬─────┘
            │ (1) "I need 4 executors x 2 cores, 4g each"
            │ (3) reply: "granted, they are starting"
       ┌────▼─────┐
       │ Cluster  │  ← resource negotiator
       │ Manager  │    (standalone / YARN / Mesos / K8s)
       └────┬─────┘
            │ (2) launch the executor JVMs
     ┌──────┴────────┐          every executor is launched with
┌────▼─────┐    ┌────▼─────┐    the driver's address, so (3b) it
│  Node 1  │    │  Node 2  │    calls the driver itself:
│┌────────┐│    │┌────────┐│    "executor 3 ready on host:port"
││Executor││    ││Executor││  ← one JVM each; its 2 cores
││Executor││    ││Executor││    = 2 tasks at a time
│└────────┘│    │└────────┘│
└──────────┘    └──────────┘  ← worker machines (nodes)

 (4) Driver  ---> Executors : ships the code, then the tasks
 (5) Executors ---> Driver  : results + success / failure
 (6) Driver ---> Cluster Manager : "finished, release them"
     the Cluster Manager kills the executor JVMs, cores freed
```

| Step | What happens | Who starts it |
|------|--------------|---------------|
| **1** | The driver (holding the `SparkSession`) asks the cluster manager for resources: *N executors, C cores each, M memory each.* | Driver |
| **2** | The cluster manager finds free capacity on the worker nodes and **launches executor JVMs** there, handing each one the driver's address. | Cluster manager |
| **3** | The manager confirms the allocation to the driver — and each executor, once alive, **registers itself directly with the driver** ("ready, N cores free, reach me at host:port"). That registration, not the manager, is how the driver learns where its executors are. | Cluster manager / executors |
| **4** | The driver **ships what the executors need to run your logic** — your serialised functions/lambdas, plus anything passed with `--py-files` / `--jars` (the full list of what travels is in section 5) — then splits the job into **stages → tasks** and hands one task per free core. | Driver |
| **5** | Each core processes **one partition** per task and reports back — the value/rows for an action, or a failure. | Executors |
| **6** | When the application ends (`spark.stop()`, script exits, or a failure), the driver tells the cluster manager to **release the resources**; the manager tears the executor JVMs down. | Driver → cluster manager |

Two things worth burning in:

- **Executors are per-application, not shared.** Another submitted program gets its own executor JVMs. That is why
  step 6 matters: unused resources held by an idle notebook are cores nobody else can use.
- **The driver never touches the data** (unless you call `collect()`, `show()`, `toPandas()` — those pull rows *to*
  the driver, which is how drivers run out of memory).

## 4. Executors vs cores — "is it 4 executors or 2?"

The video says both *"4 executors, 2 cores each = 8 cores"* and *"it created 2 executors per node"*, which sounds
contradictory. It isn't — the numbers count different things:

```text
┌──────────────────┐    ┌──────────────────┐
│      Node 1      │    │      Node 2      │
│ ┌──────┐┌──────┐ │    │ ┌──────┐┌──────┐ │
│ │exec 1││exec 2│ │    │ │exec 3││exec 4│ │  ← 4 executor
│ │ core ││ core │ │    │ │ core ││ core │ │    JVMs in total
│ │ core ││ core │ │    │ │ core ││ core │ │  ← 2 cores each
│ └──────┘└──────┘ │    │ └──────┘└──────┘ │
└──────────────────┘    └──────────────────┘

 2 nodes x 2 executors = 4 executors x 2 cores = 8 cores
 8 cores = 8 tasks at the same time = 8 partitions at once
```

| Count in this example | Where the number comes from |
|---|---|
| **2 nodes** | whatever machines the cluster happens to have free — you never ask for nodes. |
| **4 executors**, 2 per node | `--num-executors 4` is the **total**, not per node; the cluster manager decides how to spread them. |
| **2 cores** each, **8** in total | `--executor-cores 2`, once per executor. |
| **8 tasks** at a time | one task per core, one partition per task. |

So with **20 partitions and 8 cores**, Spark runs 8 tasks, then 8, then 4 — three *waves*. This is the same
arithmetic as chapter 7's "20 partitions, 2 cores → 10 waves". Partitions are the work; cores are the workers;
waves are what happens when there is more work than workers.

> ⚠️ `--num-executors` exists on YARN and Kubernetes. On **standalone** Spark you instead say
> `--total-executor-cores 8 --executor-cores 2` and Spark divides that into 4 executors. Same outcome, different dial.
> With **dynamic allocation** (`spark.dynamicAllocation.enabled=true`) you don't fix the number at all — Spark adds
> executors when tasks queue up and gives them back when they idle.

## 5. "It copies the Python program to every executor" — what actually travels?

Step 4 says the driver "copies the program to the executors". Precisely, three different things move:

| What moves | How | Note |
|---|---|---|
| **Your code** | Serialised with each task (closures/lambdas), plus any file you passed with `--py-files` / `--jars` / `--files`, which the cluster manager distributes to every executor before it starts. | This is why an executor can run *your* logic at all. |
| **The plan** | The driver's optimised physical plan, sliced into **stages** (cut at every shuffle) and then into **tasks** (one per partition). | Executors never plan; they only execute. |
| **Small lookup data** | `broadcast(df)` / broadcast variables — the driver sends one read-only copy to each executor. | The trick behind broadcast joins. |

**Your big data does *not* travel through the driver.** Each executor opens the source itself (S3/HDFS/local path)
and reads only its own partitions. The only data crossing the network between executors is **shuffle** data
(chapter 7), and the only data reaching the driver is what an action asks for.

## 6. "Based on the status — success or failure" — what does that mean?

Spark reports status at four nested levels:

```text
 application  (your whole submitted program)
   └── job     (one per action: show(), count(), write() ...)
        └── stage   (a chunk of the job between two shuffles)
             └── task   (one partition, on one core)
```

**A task fails** when the code running on that partition blows up. Real causes you will hit:

| Failure | Typical cause |
|---|---|
| Exception in your logic | dividing by zero, casting `"abc"` to int, key missing |
| Bad input data | corrupt record / wrong delimiter with `FAILFAST` (chapter 8) |
| `OutOfMemoryError` | a partition too big, a skewed key, `collect()` of a huge DataFrame |
| Executor lost | the JVM was killed (OOM killer, node reboot, spot/preemptible instance reclaimed) |
| I/O error | file deleted mid-read, permission denied, network timeout |

**What Spark does about it — it retries first.** A failed task is re-run on another executor up to
`spark.task.maxFailures` attempts (**default 4**). This is why a flaky node usually does not kill your job. If the
task still fails on the last attempt:

```text
 task fails 4x  ->  stage fails  ->  job fails  ->  app FAILED
```

The driver then aborts the job, prints the stack trace, marks the application **FAILED**, and (step 6) still asks
the cluster manager to release the executors. **Success** is the mirror image: every task of every stage finished,
the action returned its rows / the `write()` committed its files, and the application ends **SUCCEEDED**.

| Where you see the verdict | Client mode | Cluster mode |
|---|---|---|
| Stack trace | in your terminal / notebook output cell | in the cluster's driver log |
| Exit code | `spark-submit` returns non-zero on failure (`echo $?`) — this is what Airflow/cron checks | the launcher reports the final app status (on YARN it waits unless `spark.yarn.submit.waitAppCompletion=false`) |
| Web UI | `localhost:4040` while alive; the History Server afterwards | cluster manager UI (YARN RM / K8s / Spark Master) |

Practical habit: when a job fails, read the **first** failed task in the Spark UI's *Stages* tab, not the last error
in the console — the console usually shows the driver giving up, while the real cause is in the task.

## 7. The resource manager (a.k.a. cluster manager)

### "It is responsible for the driver program, not the SparkSession" — is that a mistake?

**No — it is a precision point, and a useful one.** A cluster manager deals with **operating-system processes**
(and the memory/cores they hold). The `SparkSession` is not a process: it is a **Python/Scala object living inside
the driver process**, your handle for building DataFrames.

```text
┌────────────────────────────────────────────┐
│ DRIVER PROCESS                             │  ← the cluster
│  ┌──────────────────────────────────────┐  │    manager sees
│  │ SparkSession OBJECT                  │  │    only THIS box
│  │  ┌────────────────────────────────┐  │  │
│  │  │ SparkContext                   │  │  │  ← your code sees
│  │  │  asks for / releases executors │  │  │    the inner ones
│  │  └────────────────────────────────┘  │  │    (spark.read,
│  └──────────────────────────────────────┘  │     spark.sql ...)
└────────────────────────────────────────────┘
```

Saying "the cluster manager talks to the driver" is correct. Saying "…to the SparkSession" is shorthand — harmless
in conversation, wrong when you are debugging, because in **cluster mode the manager launches the driver process
itself** (section 8), and it never knows or cares which objects live inside it.

### Beyond the landlord job description (section 0b), three things worth knowing

- It **queues** applications. Ask for more than the cluster has free and your app waits — the classic "my job is
  ACCEPTED but not running" on YARN — instead of failing.
- It **replaces executors that die**, so losing a node is usually a hiccup, not a failed job (section 6).
- In **cluster** deploy mode it launches the **driver** too, and can be configured to restart it (section 8).

### The four types

| Cluster manager | What it is | Where you meet it |
|---|---|---|
| **Standalone** | Spark's own built-in manager: a `Master` process + `Worker` processes. Simple, Spark-only. | the [docker-images/](docker-images/) cluster in this repo (1 master, 2 workers) |
| **YARN** | Hadoop's resource manager — shares the cluster between Spark, Hive, MapReduce… | classic on-prem / EMR Hadoop clusters; still the most common in enterprises |
| **Mesos** | A general datacenter resource manager. **Deprecated in Spark 3.2 and removed in Spark 4.0** — know the name, don't learn it. | legacy systems only |
| **Kubernetes** | Containerised: the driver and every executor is a **pod**. | modern cloud deployments |

Two entries that belong on the same mental shelf:

- **`local[*]`** (these notebooks) is *not* a cluster manager. There is no manager, no network, no separate
  executor: driver and executor live in **one JVM**, and `*` means "use every core on this machine".
- **Databricks / EMR / Glue** hide the manager behind their own control plane, but the story is unchanged: a driver
  node plus executor JVMs on worker nodes.

## 8. Deployment modes: client vs cluster

Both modes run the *same* program with the *same* executors. The one difference:

> **Where does the driver process run — on the machine that submitted, or inside the cluster?**

### Client mode (`--deploy-mode client`, the default)

```text
 ┌────────────────────┐
 │ YOUR MACHINE       │
 │  spark-submit      │
 │   └─> Driver       │  ← the SparkSession lives HERE,
 └──────────┬─────────┘    and must stay alive until the end
            │ asks for resources, then sends tasks
 ┌──────────▼─────────┐
 │ CLUSTER            │
 │  Cluster Manager   │
 │  Node 1: exec exec │  ← only executors run in the cluster
 │  Node 2: exec exec │
 └────────────────────┘
```

Everything described in sections 3–6 was client mode: **your machine keeps the driver**, so you see the logs live
and get results straight back. This is what a notebook does, and it is why an interrupted VPN or a closed lid kills
a running job.

### Cluster mode (`--deploy-mode cluster`)

```text
 ┌────────────────────┐
 │ YOUR MACHINE       │
 │  spark-submit      │  ← uploads the program, then its
 │  (may now exit)    │    job is DONE: you can log off
 └──────────┬─────────┘
            │ "run this program for me"
 ┌──────────▼─────────┐
 │ CLUSTER            │
 │  Cluster Manager   │  ← launches the driver too
 │  Node 1: DRIVER    │  ← the SparkSession lives HERE,
 │          exec exec │    inside the cluster
 │  Node 2: exec exec │
 └────────────────────┘
```

The client uploads the program to the cluster manager and **its only job was to submit** — after that it may exit,
you may shut your laptop, and the application keeps running because the driver is a process *inside the cluster*.
When the run ends, the driver reports the final status to the cluster manager, which then tears down the driver and
the executors together.

> 🧹 **Clearing up the wording in the video:** "the driver sits inside an executor" is not literally true.
> In cluster mode the driver is its **own separate process** running **on a worker node** — a sibling of the
> executors, not inside one. It just *looks* like another box on a node in the drawing. Names it goes by:
> the **ApplicationMaster container** on YARN, a **driver pod** on Kubernetes, a driver process on a Worker in
> standalone mode. And note it consumes cluster resources too: `--driver-memory` / `--driver-cores` come out of the
> cluster in cluster mode, out of your own machine in client mode.

### Side by side

| | **Client mode** | **Cluster mode** |
|---|---|---|
| Driver runs on | the submitting machine (laptop, edge node, notebook kernel) | a node **inside** the cluster |
| After submit, the client… | must stay connected until the end | can exit immediately ("fire and forget") |
| Logs / stack traces | stream to your terminal or notebook cell | live in the cluster (`yarn logs -applicationId …`, `kubectl logs`, manager UI) |
| `collect()` / `show()` brings rows to | your machine's memory | the driver node's memory in the cluster |
| Driver ↔ executor network | across the office/VPN link — chatty and fragile | inside the datacenter — fast and stable |
| Good for | interactive work, notebooks, `spark-shell`, debugging | production and scheduled jobs (Airflow, cron, Databricks jobs) |
| Driver failure | your process dies; nothing restarts it | the manager can be configured to restart it |

And the third thing that is *not* a deploy mode: **local mode** (`master("local[*]")`, this course) — one JVM,
no cluster manager, no network. Perfect for learning, useless for scale.

In [22]:
# Which mode am I in right now? Ask the session instead of guessing.
mode = spark.conf.get("spark.submit.deployMode", "unset")
master = spark.sparkContext.master

if master.startswith("local"):
    print(f"master={master!r} -> LOCAL mode: driver + executor in one JVM, no cluster manager, deployMode reported as {mode!r}")
else:
    print(f"master={master!r}, deploy mode={mode!r}")

# a few resource settings the driver asked for (or defaulted to)
for key in [
    "spark.master",
    "spark.app.name",
    "spark.driver.memory",
    "spark.executor.memory",
    "spark.executor.cores",
    "spark.executor.instances",
    "spark.dynamicAllocation.enabled",
    "spark.task.maxFailures",
    "spark.sql.shuffle.partitions",
]:
    try:
        # a key that was never set raises, unless Spark itself defines a default for it
        print(f"{key:35s} = {spark.conf.get(key)}")
    except Exception:
        print(f"{key:35s} = <not set: Spark's built-in default applies>")

master='local[*]' -> LOCAL mode: driver + executor in one JVM, no cluster manager, deployMode reported as 'client'
spark.master                        = local[*]
spark.app.name                      = Spark Architecture Notes
spark.driver.memory                 = <not set: Spark's built-in default applies>
spark.executor.memory               = <not set: Spark's built-in default applies>
spark.executor.cores                = <not set: Spark's built-in default applies>
spark.executor.instances            = <not set: Spark's built-in default applies>
spark.dynamicAllocation.enabled     = <not set: Spark's built-in default applies>
spark.task.maxFailures              = <not set: Spark's built-in default applies>
spark.sql.shuffle.partitions        = 200


## 9. `spark-submit` cheat sheet — flag → story

| Flag | Which part of the story it controls |
|---|---|
| `--master yarn` / `k8s://…` / `spark://host:7077` / `local[*]` | **which cluster manager** the driver negotiates with (section 7) |
| `--deploy-mode client\|cluster` | **where the driver runs** (section 8) |
| `--num-executors 4` | how many executor JVMs to ask for in step 1 (YARN/K8s) |
| `--executor-cores 2` | task slots per executor → parallelism (section 4) |
| `--executor-memory 4g` | heap size of each executor JVM (section 0a) |
| `--driver-memory 2g`, `--driver-cores 1` | the driver's own JVM — raise it before `collect()`ing a lot |
| `--total-executor-cores 8` | the standalone-mode way to size the app |
| `--py-files utils.zip`, `--jars a.jar`, `--files config.yaml` | extra things shipped to every executor (section 5) |
| `--conf spark.dynamicAllocation.enabled=true` | let Spark grow/shrink the executor count on demand |
| `--name daily_sales` | the application name in the manager UI and History Server |
| `app.py arg1 arg2` | your program and its arguments — always last |

Rule of thumb for sizing: **many small executors** parallelise better and lose less when one dies; **few fat
executors** cache more and shuffle less. The usual compromise is 2–5 cores per executor, never 1 giant executor
per node.

## 10. Where this course sits

The vocabulary is all in section 0 — this last section only places *your* three setups on the map.

| Setup | Cluster manager | Driver | Executors |
|---|---|---|---|
| `master("local[*]")` — chapters 1-10 | none (local mode) | your Python + JVM process | none: the same JVM does the work, `*` cores |
| [docker-images/](docker-images/) | standalone (1 master + 2 workers) | the Jupyter container (client mode) | JVMs inside the worker containers |
| Databricks Community (chapter 6 tip) | Databricks' own | the cluster's driver node | single-node cluster: the driver's own cores |

### Watch it happen

Open the Spark UI (`localhost:4040`, see [jup_Note_url.txt](jup_Note_url.txt)) while a job runs — it is this whole
note, live:

- **Executors** tab → every executor JVM, its cores, its memory, tasks completed/failed — sections 3 and 4.
- **Environment** tab → every `--conf` and flag the driver was launched with — section 9.
- **Jobs / Stages** tabs → the job → stage → task tree, with the wave pattern of section 4 and any retries from
  section 6.

In [23]:
# Free the resources — step 6 of the lifecycle, done by hand.
# The driver tells the cluster manager it is finished; executors get torn down.
spark.stop()
print("SparkSession stopped -> executors released, Spark UI on :4040 is gone")

SparkSession stopped -> executors released, Spark UI on :4040 is gone
